In [ ]:
import numpy as np
import torch
from torch import nn, Tensor
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Dataset
from matplotlib import pyplot as plt
import rasterio
from sklearn.preprocessing import StandardScaler
import seaborn as sns
from typing import Literal, Callable, NamedTuple
import pandas as pd

from constants import SEED, TILE_SIZE
from grad_descent_train_eval import ExperimentRunner

np.random.seed(SEED)
torch.manual_seed(SEED)

# render figures as crisp vector graphics
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('svg')
except ImportError:
    pass

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device.")

Using mps device.


In [ ]:
class FireLinearParams(NamedTuple):
    tile_width: int
    tile_height: int
    num_categories: int = 1

    def __str__(self):
        return f"Lin(w={self.tile_width}, h={self.tile_height}, " + \
            f"out={self.num_categories})"


class FireLinear(nn.Module):
    def __init__(self, args: FireLinearParams):
        super().__init__()
        self.args = args
        self.net = nn.Sequential(
            nn.Flatten(start_dim=1, end_dim=2),
            nn.Linear(args.tile_width * args.tile_height, args.num_categories),
        )

    def forward(self, X: Tensor):
        """
        Input: (batch, tile_width, tile_height, embed_dim)
        Output: (batch, tile_width, tile_height, num_categories)
        """
        return self.net(X)


In [ ]:
def load_surface_fuel_data() -> tuple[DataLoader, DataLoader, DataLoader]:
    with rasterio.open("data/feature_and_class_tiled.tif") as src:
        data = src.read()

    X = data[:-1]
    y = data[-1]

    # 1 = train, 2 = validation, 3 = test
    PATTERN = np.array([[1, 1, 1],
                    [1, 2, 3]], dtype="uint8")

    rows, cols = y.shape
    block_r = np.arange(rows) // 512
    block_c = np.arange(cols) // 512
    split = PATTERN[block_r[:, None] % PATTERN.shape[0],
                block_c[None, :] % PATTERN.shape[1]]

    train_dataset = TensorDataset(X[:, split == 1].T, y[split == 1])
    val_dataset = TensorDataset(X[:, split == 2].T, y[split == 2])
    test_dataset = TensorDataset(X[:, split == 3].T, y[split == 3])

    return DataLoader(train_dataset), DataLoader(val_dataset), DataLoader(test_dataset)

In [ ]:
full_train_loader, full_val_loader, full_test_loader = load_surface_fuel_data()

In [ ]:
runner1_1 = (
    ExperimentRunner(FireLinear)
        .with_epochs(100)
        .with_data(full_train_loader, full_val_loader)
        .add_experiments(
            FireLinearParams(
                tile_width=TILE_SIZE,
                tile_height=TILE_SIZE,
                num_categories=999999999999,  # TODO(ianfisk):
            )
        ).with_plots()
)

runner_results1_1 = runner1_1.run(verbosity=1)
runner_results1_1